In [1]:
import math
import pandas as pd

GLICKO_SCALE = 173.7178
BASE_RATING = 1500.0
BASE_RD = 350.0
BASE_SIGMA = 0.06
TAU = 0.5
VOLATILITY_EPSILON = 1e-6

FORM_WINDOW = 3
FORM_NUDGE_MAX = 8.0
METHOD_NUDGES = {
    'ko_tko': 3.0,
    'submission': 2.0,
    'decision': -1.0,
    'other': 0.0,
}
OPPONENT_FORM_NUDGE_MAX = 2.0
TOTAL_CUSTOM_NUDGE_CAP = 10.0

ufc_fights = pd.read_csv('../../scraping/output/ufc_fights.csv')
ufc_fights = ufc_fights.drop(
    columns=['fight_id', 'event_id', 'event_url', 'fight_url', 'fight_order', 'time', 'scraped_at']
)
ufc_fights = ufc_fights.reset_index()
ufc_fights = ufc_fights.sort_index(ascending=False)

ufc_fights['result'] = ufc_fights['result'].astype(str).str.lower().apply(
    lambda x: 'draw' if 'draw' in x else ('nc' if 'nc' in x or 'no contest' in x else ('win' if 'win' in x else 'unknown'))
)
ufc_fights['method'] = ufc_fights['method'].astype(str).str.upper().apply(
    lambda x: 'ko_tko' if x.startswith('KO/TKO') else ('submission' if x.startswith('SUB') else ('decision' if 'DEC' in x else 'other'))
)
ufc_fights['round'] = pd.to_numeric(ufc_fights['round'], errors='coerce')
ufc_fights['fighter_1'] = ufc_fights['fighter_1'].astype(str).str.strip()
ufc_fights['fighter_2'] = ufc_fights['fighter_2'].astype(str).str.strip()
ufc_fights['weight_class'] = ufc_fights['weight_class'].astype(str).str.strip()

fighters = {}
ufc_fights.head()


,index,event_name,event_date,fighter_1,fighter_2,result,weight_class,method,round
8633,8633,UFC 3: The American Dream,"September 09, 1994",Keith Hackney,Emmanuel Yarborough,win,Open Weight,ko_tko,1
8632,8632,UFC 3: The American Dream,"September 09, 1994",Ken Shamrock,Christophe Leninger,win,Open Weight,ko_tko,1
8631,8631,UFC 3: The American Dream,"September 09, 1994",Harold Howard,Roland Payne,win,Open Weight,ko_tko,1
8630,8630,UFC 3: The American Dream,"September 09, 1994",Royce Gracie,Kimo Leopoldo,win,Open Weight,submission,1
8629,8629,UFC 3: The American Dream,"September 09, 1994",Ken Shamrock,Felix Lee Mitchell,win,Open Weight,submission,1


In [2]:
def load_fighter(name, weight):
    if name not in fighters:
        fighters[name] = {
            'rating': BASE_RATING,
            'RD': BASE_RD,
            'sigma': BASE_SIGMA,
            'weight_class': weight,
            'recent_scores': [],
            'fight_count': 0,
        }


In [3]:
def g(phi):
    return 1 / math.sqrt(1 + (3 * phi**2) / (math.pi**2))


def expected_score(mu, mu_j, phi_j):
    return 1 / (1 + math.exp(-g(phi_j) * (mu - mu_j)))


def volatility_objective(x, delta, phi, v, a, tau):
    exp_x = math.exp(x)
    numerator = exp_x * (delta**2 - phi**2 - v - exp_x)
    denominator = 2 * (phi**2 + v + exp_x) ** 2
    return (numerator / denominator) - ((x - a) / (tau**2))


def update_sigma(phi, delta, v, sigma, tau=TAU, epsilon=VOLATILITY_EPSILON):
    a = math.log(sigma**2)
    A = a

    if delta**2 > phi**2 + v:
        B = math.log(delta**2 - phi**2 - v)
    else:
        k = 1
        while volatility_objective(a - (k * tau), delta, phi, v, a, tau) < 0:
            k += 1
        B = a - (k * tau)

    f_A = volatility_objective(A, delta, phi, v, a, tau)
    f_B = volatility_objective(B, delta, phi, v, a, tau)

    while abs(B - A) > epsilon:
        C = A + ((A - B) * f_A / (f_B - f_A))
        f_C = volatility_objective(C, delta, phi, v, a, tau)

        if f_C * f_B <= 0:
            A = B
            f_A = f_B
        else:
            f_A /= 2

        B = C
        f_B = f_C

    return math.exp(A / 2)


def recent_form_factor(fighter, window=FORM_WINDOW):
    recent_scores = fighter.get('recent_scores', [])[-window:]
    if not recent_scores:
        return 0.0

    average_score = sum(recent_scores) / len(recent_scores)
    return (average_score - 0.5) * 2


def apply_custom_nudge(rating, fighter, opponent, result, method):
    if result == 0.5:
        return rating, 0.0

    direction = 1 if result > 0.5 else -1
    form_nudge = FORM_NUDGE_MAX * recent_form_factor(fighter)
    method_nudge = METHOD_NUDGES.get(method, METHOD_NUDGES['other'])
    opponent_form_nudge = OPPONENT_FORM_NUDGE_MAX * recent_form_factor(opponent)

    total_nudge = direction * (form_nudge + method_nudge + opponent_form_nudge)
    total_nudge = max(-TOTAL_CUSTOM_NUDGE_CAP, min(TOTAL_CUSTOM_NUDGE_CAP, total_nudge))

    return rating + total_nudge, total_nudge


def glicko2_update(fighter, opponent, score):
    mu = (fighter['rating'] - BASE_RATING) / GLICKO_SCALE
    phi = fighter['RD'] / GLICKO_SCALE
    sigma = fighter['sigma']

    mu_j = (opponent['rating'] - BASE_RATING) / GLICKO_SCALE
    phi_j = opponent['RD'] / GLICKO_SCALE

    g_phi_j = g(phi_j)
    expected = expected_score(mu, mu_j, phi_j)
    v = 1 / (g_phi_j**2 * expected * (1 - expected))
    delta = v * g_phi_j * (score - expected)

    sigma_new = update_sigma(phi, delta, v, sigma)
    phi_star = math.sqrt(phi**2 + sigma_new**2)
    phi_new = 1 / math.sqrt((1 / phi_star**2) + (1 / v))
    mu_new = mu + (phi_new**2) * g_phi_j * (score - expected)

    return {
        'rating': BASE_RATING + (GLICKO_SCALE * mu_new),
        'RD': GLICKO_SCALE * phi_new,
        'sigma': sigma_new,
    }


def update_rating(fighter1, fighter2, result, method, round_):
    updated_fighter1 = {**fighter1, **glicko2_update(fighter1, fighter2, result)}
    updated_fighter2 = {**fighter2, **glicko2_update(fighter2, fighter1, 1 - result)}

    updated_fighter1['rating'], nudge1 = apply_custom_nudge(
        updated_fighter1['rating'], fighter1, fighter2, result, method
    )
    updated_fighter2['rating'], nudge2 = apply_custom_nudge(
        updated_fighter2['rating'], fighter2, fighter1, 1 - result, method
    )

    return updated_fighter1, updated_fighter2, nudge1, nudge2


def update_recent_history(fighter, score):
    fighter['recent_scores'] = fighter.get('recent_scores', []) + [score]
    fighter['recent_scores'] = fighter['recent_scores'][-FORM_WINDOW:]
    fighter['fight_count'] = fighter.get('fight_count', 0) + 1


In [4]:
score_map = {'win': 1.0, 'draw': 0.5}
tracked_columns = [
    'fighter_1_rating_start',
    'fighter_2_rating_start',
    'fighter_1_rating_end',
    'fighter_2_rating_end',
    'fighter_1_RD_start',
    'fighter_2_RD_start',
    'fighter_1_RD_end',
    'fighter_2_RD_end',
    'fighter_1_sigma_start',
    'fighter_2_sigma_start',
    'fighter_1_sigma_end',
    'fighter_2_sigma_end',
    'fighter_1_custom_nudge',
    'fighter_2_custom_nudge',
]

for column in tracked_columns:
    ufc_fights[column] = None

for idx, row in ufc_fights.iterrows():
    fighter1_name = row['fighter_1']
    fighter2_name = row['fighter_2']
    weight = row['weight_class']
    result_name = row['result']
    method = row['method']
    round_ = row['round']

    load_fighter(fighter1_name, weight)
    load_fighter(fighter2_name, weight)
    if weight != 'Catch Weight':
        fighters[fighter1_name]['weight_class'] = weight
        fighters[fighter2_name]['weight_class'] = weight

    fighter1 = fighters[fighter1_name]
    fighter2 = fighters[fighter2_name]

    ufc_fights.at[idx, 'fighter_1_rating_start'] = fighter1['rating']
    ufc_fights.at[idx, 'fighter_2_rating_start'] = fighter2['rating']
    ufc_fights.at[idx, 'fighter_1_RD_start'] = fighter1['RD']
    ufc_fights.at[idx, 'fighter_2_RD_start'] = fighter2['RD']
    ufc_fights.at[idx, 'fighter_1_sigma_start'] = fighter1['sigma']
    ufc_fights.at[idx, 'fighter_2_sigma_start'] = fighter2['sigma']

    if result_name == 'nc':
        ufc_fights.at[idx, 'fighter_1_rating_end'] = fighter1['rating']
        ufc_fights.at[idx, 'fighter_2_rating_end'] = fighter2['rating']
        ufc_fights.at[idx, 'fighter_1_RD_end'] = fighter1['RD']
        ufc_fights.at[idx, 'fighter_2_RD_end'] = fighter2['RD']
        ufc_fights.at[idx, 'fighter_1_sigma_end'] = fighter1['sigma']
        ufc_fights.at[idx, 'fighter_2_sigma_end'] = fighter2['sigma']
        ufc_fights.at[idx, 'fighter_1_custom_nudge'] = 0.0
        ufc_fights.at[idx, 'fighter_2_custom_nudge'] = 0.0
        continue

    result = score_map.get(result_name, 0.5)
    updated_fighter1, updated_fighter2, nudge1, nudge2 = update_rating(
        fighter1, fighter2, result, method, round_
    )

    if weight != 'Catch Weight':
        updated_fighter1['weight_class'] = weight
        updated_fighter2['weight_class'] = weight
    else:
        updated_fighter1['weight_class'] = fighters[fighter1_name]['weight_class']
        updated_fighter2['weight_class'] = fighters[fighter2_name]['weight_class']
    update_recent_history(updated_fighter1, result)
    update_recent_history(updated_fighter2, 1 - result)

    fighters[fighter1_name] = updated_fighter1
    fighters[fighter2_name] = updated_fighter2

    ufc_fights.at[idx, 'fighter_1_rating_end'] = updated_fighter1['rating']
    ufc_fights.at[idx, 'fighter_2_rating_end'] = updated_fighter2['rating']
    ufc_fights.at[idx, 'fighter_1_RD_end'] = updated_fighter1['RD']
    ufc_fights.at[idx, 'fighter_2_RD_end'] = updated_fighter2['RD']
    ufc_fights.at[idx, 'fighter_1_sigma_end'] = updated_fighter1['sigma']
    ufc_fights.at[idx, 'fighter_2_sigma_end'] = updated_fighter2['sigma']
    ufc_fights.at[idx, 'fighter_1_custom_nudge'] = nudge1
    ufc_fights.at[idx, 'fighter_2_custom_nudge'] = nudge2


In [6]:
fighters_df = pd.DataFrame(
    [
        {
            'fighter_name': fighter_name,
            'rating': fighter_data['rating'],
            'RD': fighter_data['RD'],
            'sigma': fighter_data['sigma'],
            'weight_class': fighter_data['weight_class'],
            'fight_count': fighter_data['fight_count'],
            'recent_form_factor': recent_form_factor(fighter_data),
        }
        for fighter_name, fighter_data in fighters.items()
    ]
).sort_values(['rating', 'RD'], ascending=[False, True]).reset_index(drop=True)

fighters_df.head(30)


,fighter_name,rating,RD,sigma,weight_class,fight_count,recent_form_factor
0,Jon Jones,2460.009855,93.434754,0.059984,Heavyweight,23,1.000000
1,Ilia Topuria,2394.429727,138.413140,0.059996,Lightweight,9,1.000000
2,Islam Makhachev,2365.782108,101.432605,0.059990,Welterweight,18,1.000000
3,Georges St-Pierre,2327.919381,96.767880,0.059994,Middleweight,22,1.000000
4,Shavkat Rakhmonov,2313.802761,148.392632,0.059996,Welterweight,7,1.000000
5,Khabib Nurmagomedov,2303.373432,117.584903,0.059989,Lightweight,13,1.000000
6,Khamzat Chimaev,2280.772668,126.670092,0.060001,Middleweight,9,1.000000
7,Alex Pereira,2280.184305,118.001188,0.060006,Light Heavyweight,12,0.333333
8,Daniel Cormier,2279.016460,111.877641,0.059999,Heavyweight,14,-0.333333
9,Alexander Volkanovski,2273.409405,95.393059,0.059993,Featherweight,18,0.333333
